# Domain-Adaptable Speech-to-Text — End-to-End Training

This notebook runs the full pipeline:
1. Clone repo / mount Drive for checkpoint persistence
2. Install dependencies
3. Generate synthetic domain prompts
4. Synthesize audio (TTS)
5. Augment with noise
6. Prepare HF dataset (mel-spectrogram features + tokenized labels)
7. LoRA fine-tune Whisper-small
8. Evaluate WER: baseline vs fine-tuned, clean vs noisy



In [87]:
# 0. Mount Google Drive so checkpoints/data survive Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/foundry-speech-to-text'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/foundry-speech-to-text


In [88]:
# 1. Clone the repo
!git clone https://github.com/Yamini1727/foundry-speech-to-text.git repo
%cd repo

fatal: destination path 'repo' already exists and is not an empty directory.
/content/drive/MyDrive/foundry-speech-to-text/repo


In [89]:
%cd /content/drive/MyDrive/foundry-speech-to-text/repo
!git pull

/content/drive/MyDrive/foundry-speech-to-text/repo
Already up to date.


In [90]:
!pwd

/content/drive/MyDrive/foundry-speech-to-text/repo


In [91]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [92]:
!pip install -q -U torchao

In [93]:
# 2b. Confirm ffmpeg is available (needed to convert edge-tts mp3 output -> wav)
!ffmpeg -version | head -n 1

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [94]:
# 3. Generate prompts
!python data/generate_prompts.py --output data/prompts.json --n_general 150 --n_domain 200 --n_generic_obs 100

Generated 450 prompts -> data/prompts.json
  general: 150, domain: 200, generic_obs: 100


In [95]:
!cat data/synthesize_audio.py!cat data/synthesize_audio.py

cat: 'data/synthesize_audio.py!cat': No such file or directory
"""
synthesize_audio.py

Converts text prompts (from generate_prompts.py) into audio files using
edge-tts: a free, open wrapper around Microsoft Edge's neural TTS voices.
No API key, no account, no paid tier.

Why edge-tts instead of Coqui/XTTS: XTTS is a heavy model whose dependencies
(older transformers internals) conflict with the modern transformers version
we need for Whisper fine-tuning in the same environment -- installing both
in one Colab runtime breaks one or the other. edge-tts has no ML framework
dependency at all (it just calls Microsoft's TTS service over the network),
so it can't conflict with anything else we install for training.

Bonus: edge-tts ships many high-quality neural voices across accents
out of the box (US / UK / Indian / Australian English) which gives us
genuine voice + accent diversity for free -- arguably more useful for
robustness than XTTS's default speaker set, and directly relevant if the

In [96]:
!git fetch origin
!git reset --hard origin/main

HEAD is now at 80304d2 Update and rename evaluate.py to run_evaluation.py


In [97]:
!sed -n '104,114p' training/finetune_lora.py

    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=data_collator,
        processing_class=processor.feature_extractor,
    )



In [98]:
!ls training/

finetune_lora.py  prepare_dataset.py  __pycache__  run_evaluation.py


In [99]:
!head -40 data/synthesize_audio.py

"""
synthesize_audio.py

Converts text prompts (from generate_prompts.py) into audio files using
edge-tts: a free, open wrapper around Microsoft Edge's neural TTS voices.
No API key, no account, no paid tier.

Why edge-tts instead of Coqui/XTTS: XTTS is a heavy model whose dependencies
(older transformers internals) conflict with the modern transformers version
we need for Whisper fine-tuning in the same environment -- installing both
in one Colab runtime breaks one or the other. edge-tts has no ML framework
dependency at all (it just calls Microsoft's TTS service over the network),
so it can't conflict with anything else we install for training.

Bonus: edge-tts ships many high-quality neural voices across accents
out of the box (US / UK / Indian / Australian English) which gives us
genuine voice + accent diversity for free -- arguably more useful for
robustness than XTTS's default speaker set, and directly relevant if the
target usage involves Indian-English speakers.

Meant to run o

In [100]:
# 4. Synthesize audio via TTS
!python data/synthesize_audio.py --prompts data/prompts.json --output_dir data/audio_raw

Synthesized 900 audio files -> data/audio_raw
Manifest -> data/audio_raw/manifest.json


In [101]:
# 5. Augment with machinery/industrial noise (ESC-50, free CC-licensed dataset)
!python data/augment_noise.py --manifest data/audio_raw/manifest.json --output_dir data/audio_augmented

Found 160 machinery/industrial-ish noise clips
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/drive/MyDrive/foundry-speech-to-text/repo/ESC-50/audio/5-207811-A-35.wav had to be resampled from 44100 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/drive/MyDrive/foundry-speech-to-text/repo/ESC-50/audio/5-232272-A-44.wav had to be resampled from 44100 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/drive/MyDrive/foundry-speech-to-text/repo/ESC-50/audio/1-60460-A-36.wav had to be resampled from 44100 Hz to 16000 Hz. This hurt execution time.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/audiomentations/core/audio_loading_utils.py:36: UserWarning: /content/drive/MyDrive/foundry-spe

In [102]:
# 6. Prepare HF dataset (feature extraction + tokenization)
!python training/prepare_dataset.py --manifest data/audio_augmented/manifest_full.json \
    --output_dir hf_dataset --model_name openai/whisper-small

Loading manifest...
Splitting train / val / test (stratified-ish by condition via shuffle+split)...
train=1034  val=115  test=203
Test split condition breakdown: {'clean': 141, 'noisy': 62}
Loading feature extractor + tokenizer for openai/whisper-small...
Extracting features (this does the heavy lifting: mel-spectrograms + tokenization)...
Map: 100% 1034/1034 [00:46<00:00, 22.09 examples/s]
Map: 100% 115/115 [00:04<00:00, 25.73 examples/s]
Map: 100% 203/203 [00:09<00:00, 21.77 examples/s]
Saving the dataset (2/2 shards): 100% 1034/1034 [00:03<00:00, 305.44 examples/s]
Saving the dataset (1/1 shards): 100% 115/115 [00:02<00:00, 48.46 examples/s]
Saving the dataset (1/1 shards): 100% 203/203 [00:00<00:00, 441.04 examples/s]
Saved processed dataset -> hf_dataset


In [103]:
!pip show torchao | grep Version

Version: 0.18.0


In [104]:
# 7. LoRA fine-tune (the actual training step)
!python training/finetune_lora.py --dataset_dir hf_dataset --output_dir whisper-lora-foundry \
    --model_name openai/whisper-small --epochs 4 --batch_size 8

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading dataset...
Loading base model: openai/whisper-small
Loading weights: 100% 479/479 [00:00<00:00, 11006.56it/s]
Applying LoRA adapters to attention projections (q_proj, v_proj)...
trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429
Starting LoRA fine-tuning...
{'loss': '8.578', 'grad_norm': '4.954', 'learning_rate': '0.00018', 'epoch': '0.1538'}
{'loss': '4.053', 'grad_norm': '31.38', 'learning_rate': '0.00038', 'epoch': '0.3077'}
{'loss': '1.738', 'grad_norm': '1.287', 'learning_rate': '0.00058', 'epoch': '0.4615'}
{'loss': '1.204', 'grad_norm': '0.948

In [105]:
# 8. Evaluate: baseline vs fine-tuned, clean vs noisy WER
!python training/run_evaluation.py --dataset_dir hf_dataset --base_model openai/whisper-small \
    --lora_adapter whisper-lora-foundry --output results/wer_comparison.json

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
=== Evaluating BASELINE (vanilla pretrained Whisper) ===
Loading weights: 100% 479/479 [00:00<00:00, 4257.49it/s]
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://git

In [106]:
# 9. Quick sanity-check transcription on a single held-out sample
import json
with open('results/wer_comparison.json') as f:
    results = json.load(f)
for sample in results['sample_predictions'][:5]:
    print('REF:', sample['reference'])
    print('BASELINE:', sample['baseline_pred'])
    print('FINE-TUNED:', sample['finetuned_pred'])
    print('---')

REF: the server logs show no errors this morning
BASELINE:  The server logs show no errors this morning.
FINE-TUNED: the server logs show no errors this morning
---
REF: green compactability at station one is 19 point 6 percent
BASELINE:  Green Compactability at Station 1 is 19.6%.
FINE-TUNED: green compactability at station one is 19 point 6 percent
---
REF: sand temperature at station two is 37 degrees celsius
BASELINE:  Sand temperature at station 2 is 37 degrees Celsius.
FINE-TUNED: sand temperature at station two is 37 degrees celsius
---
REF: recording loss on ignition as 19 point 4 percent on station two
BASELINE:  Recording loss on ignition as 19.4% on station 2.
FINE-TUNED: recording loss on ignition as 19 point 4 percent on station two
---
REF: dead clay content reading is 11 point 2 percent
BASELINE:  Dead Clay content reading is 11.2%.
FINE-TUNED: dead clay content reading is 11 point 2 percent
---
